# Car Price Prediction - Exploratory Data Analysis

This notebook performs a comprehensive exploratory data analysis on the cars24 dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load the dataset
df = pd.read_csv('cars24-car-price-cleaned-new.csv')
print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")

: 

## 1. Dataset Overview

In [ ]:
# Display first few rows
print("First few rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nBasic Statistics:")
print(df.describe())

In [ ]:
# Check for missing values
print("Missing Values:")
missing = df.isnull().sum()
if missing.sum() == 0:
    print("No missing values found!")
else:
    print(missing[missing > 0])

## 2. Target Variable Analysis

In [ ]:
# Selling price distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['selling_price'], bins=50, color='skyblue', edgecolor='black')
axes[0].set_xlabel('Selling Price')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Selling Price')
axes[0].grid(alpha=0.3)

# Box plot
axes[1].boxplot(df['selling_price'])
axes[1].set_ylabel('Selling Price')
axes[1].set_title('Box Plot of Selling Price')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Selling Price Statistics:")
print(f"Mean: {df['selling_price'].mean():.2f}")
print(f"Median: {df['selling_price'].median():.2f}")
print(f"Std Dev: {df['selling_price'].std():.2f}")
print(f"Min: {df['selling_price'].min():.2f}")
print(f"Max: {df['selling_price'].max():.2f}")

## 3. Numerical Features Analysis

In [ ]:
# Select numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numerical columns: {numerical_cols}")

# Create distribution plots for numerical features
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.ravel()

for idx, col in enumerate(numerical_cols[:9]):
    axes[idx].hist(df[col], bins=30, color='steelblue', edgecolor='black')
    axes[idx].set_title(f'Distribution of {col}')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Correlation Analysis

In [ ]:
# Correlation matrix
correlation_matrix = df.corr(numeric_only=True)

# Correlation with target variable
target_corr = correlation_matrix['selling_price'].sort_values(ascending=False)
print("Correlation with Selling Price:")
print(target_corr)

# Plot correlation matrix
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, cbar_kws={'label': 'Correlation'})
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. Categorical Features Analysis

In [ ]:
# Identify categorical columns (excluding binary flags)
print("Unique values per column:")
for col in df.columns:
    if df[col].dtype == 'object':
        print(f"{col}: {df[col].nunique()} unique values")
        print(f"  Top values: {df[col].value_counts().head(3).to_dict()}")

## 6. Feature Relationships with Target

In [ ]:
# Top correlations with selling price
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top 4 features correlated with selling_price (excluding itself)
top_features = target_corr.drop('selling_price').head(4).index

for idx, feature in enumerate(top_features):
    row = idx // 2
    col = idx % 2
    axes[row, col].scatter(df[feature], df['selling_price'], alpha=0.6)
    axes[row, col].set_xlabel(feature)
    axes[row, col].set_ylabel('Selling Price')
    axes[row, col].set_title(f'{feature} vs Selling Price')
    axes[row, col].grid(alpha=0.3)
    
    # Add correlation coefficient
    corr = df[feature].corr(df['selling_price'])
    axes[row, col].text(0.05, 0.95, f'Corr: {corr:.3f}', 
                        transform=axes[row, col].transAxes, 
                        verticalalignment='top',
                        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## 6.5. Selling Price vs km_driven Analysis

In [ ]:
# 1. Scatter plot: Selling Price vs km_driven
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot
axes[0].scatter(df['km_driven'], df['selling_price'], alpha=0.5, s=30, color='steelblue')
axes[0].set_xlabel('KM Driven', fontsize=12)
axes[0].set_ylabel('Selling Price', fontsize=12)
axes[0].set_title('Scatter Plot: Selling Price vs KM Driven', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)

# Add correlation coefficient
corr_km = df['km_driven'].corr(df['selling_price'])
axes[0].text(0.05, 0.95, f'Correlation: {corr_km:.4f}', 
            transform=axes[0].transAxes, 
            verticalalignment='top',
            fontsize=11,
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

# 2. Bucketed Analysis: Create price buckets and analyze km_driven
price_buckets = [0, 5, 10, 15, 20, float('inf')]
bucket_labels = ['0-5', '5-10', '10-15', '15-20', '20+']
df['price_bucket'] = pd.cut(df['selling_price'], bins=price_buckets, labels=bucket_labels, right=False)

# Calculate statistics for each bucket
bucket_stats = df.groupby('price_bucket', observed=True)['km_driven'].agg(['mean', 'median', 'count']).reset_index()
bucket_stats.columns = ['Price Bucket', 'Avg KM Driven', 'Median KM Driven', 'Count']

print("Selling Price Buckets vs KM Driven Statistics:")
print(bucket_stats.to_string(index=False))

# Bar plot: Average KM driven per price bucket
x_pos = np.arange(len(bucket_stats))
bars = axes[1].bar(x_pos, bucket_stats['Avg KM Driven'], color='coral', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Selling Price Bucket (in Lakhs)', fontsize=12)
axes[1].set_ylabel('Average KM Driven', fontsize=12)
axes[1].set_title('Average KM Driven by Selling Price Bucket', fontsize=14, fontweight='bold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(bucket_stats['Price Bucket'])
axes[1].grid(axis='y', alpha=0.3)

# Add value labels on top of bars
for i, (bar, val) in enumerate(zip(bars, bucket_stats['Avg KM Driven'])):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000, 
                f'{val:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Add count labels inside bars
for i, (bar, count) in enumerate(zip(bars, bucket_stats['Count'])):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height()/2, 
                f'n={int(count)}', ha='center', va='center', fontsize=9, color='white', fontweight='bold')

plt.tight_layout()
plt.show()

# Additional insights
print("\n" + "="*60)
print("KEY INSIGHTS: Selling Price vs KM Driven")
print("="*60)
print(f"\nCorrelation between selling_price and km_driven: {corr_km:.4f}")
print(f"\nPrice Bucket Distribution:")
for _, row in bucket_stats.iterrows():
    pct = (row['Count'] / len(df)) * 100
    print(f"  {row['Price Bucket']} Lakhs: {int(row['Count'])} cars ({pct:.1f}%)")
print(f"\nKM Driven Insights:")
print(f"  Lowest avg km_driven: {bucket_stats['Avg KM Driven'].min():,.0f} (Price Bucket: {bucket_stats.loc[bucket_stats['Avg KM Driven'].idxmin(), 'Price Bucket']})")
print(f"  Highest avg km_driven: {bucket_stats['Avg KM Driven'].max():,.0f} (Price Bucket: {bucket_stats.loc[bucket_stats['Avg KM Driven'].idxmax(), 'Price Bucket']})")

## 7. Outlier Detection

In [ ]:
# Using IQR method for outlier detection
def identify_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return lower_bound, upper_bound

print("Outlier Detection (IQR Method):")
print("=" * 50)

outlier_counts = {}
for col in ['selling_price', 'km_driven', 'mileage', 'engine', 'max_power']:
    lower, upper = identify_outliers(df, col)
    outliers = df[(df[col] < lower) | (df[col] > upper)][col]
    outlier_counts[col] = len(outliers)
    print(f"{col}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.2f}%)")
    print(f"  Range: [{lower:.2f}, {upper:.2f}]")

## 8. Summary and Insights

In [ ]:
print("\n" + "="*60)
print("KEY INSIGHTS FROM EDA")
print("="*60)
print(f"\n1. Dataset Size: {len(df)} records, {len(df.columns)} features")
print(f"\n2. Target Variable (selling_price):")
print(f"   - Range: {df['selling_price'].min():.2f} to {df['selling_price'].max():.2f}")
print(f"   - Mean: {df['selling_price'].mean():.2f}")
print(f"   - Median: {df['selling_price'].median():.2f}")
print(f"\n3. Top Correlated Features:")
for i, (feat, corr) in enumerate(target_corr.drop('selling_price').head(5).items(), 1):
    print(f"   {i}. {feat}: {corr:.4f}")
print(f"\n4. Data Quality:")
print(f"   - Missing values: {df.isnull().sum().sum()}")
print(f"   - Duplicate rows: {df.duplicated().sum()}")
print(f"\n5. Numerical Features: {len(numerical_cols)}")
print(f"\n6. Next Steps for Modeling:")
print(f"   - Handle any outliers")
print(f"   - Scale numerical features")
print(f"   - Test multiple algorithms")
print(f"   - Tune hyperparameters")